# Moral Valence Analysis by Gender and Topic

Scores sentences using eMFDscore (MFD2, bag-of-words), computes moral valence per gender for each topic file in `corpus_normalization/normalised_mp_sentences_by_topic/`.

In [6]:
import pandas as pd
from pathlib import Path
from emfdscore.scoring import score_docs

in_dir = Path("../../corpus_normalization/normalised_mp_sentences_by_topic")
out_dir = Path("results_by_topic")
out_dir.mkdir(parents=True, exist_ok=True)

DICT_TYPE = "mfd2"
PROB_MAP = ""
SCORE_METHOD = "bow"
OUT_METRICS = ""

foundations = ["care", "fairness", "loyalty", "authority", "sanctity"]
eps = 1e-9

In [7]:
topic_files = sorted(in_dir.glob("*.csv"))
if not topic_files:
    raise FileNotFoundError(f"No topic CSVs found in: {in_dir.resolve()}")

print(f"Found {len(topic_files)} topic files:")
for f in topic_files:
    print(f"  {f.name}")

Found 10 topic files:
  0_covid_virus_vaccine_pandemic.csv
  1_climate_emissions_carbon_energy.csv
  2_scottish_scotland_snp_sturgeon.csv
  3_biden_trump_sanders_democratic.csv
  4_schools_school_education_teachers.csv
  5_farage_ukip_nuttall_party.csv
  6_her_she_may_brexit.csv
  7_book_books_novel_my.csv
  8_nhs_health_doctors_patients.csv
  9_eu_barnier_uk_deal.csv


In [8]:
all_results = []

for file in topic_files:
    topic = file.stem
    print(f"\n{'='*60}")
    print(f"Topic: {topic}")
    print(f"{'='*60}")

    df = pd.read_csv(file)

    if "sentence" not in df.columns or "gender" not in df.columns:
        print(f"  Skipping — missing 'sentence' or 'gender' column")
        continue

    if len(df) == 0:
        print("  Skipping — empty file")
        continue

    # eMFDscore expects first column as text
    emfd_input = pd.DataFrame({"text": df["sentence"].fillna("").astype(str)})

    scored = score_docs(emfd_input, DICT_TYPE, PROB_MAP, SCORE_METHOD, OUT_METRICS, len(emfd_input))
    scored["gender"] = df["gender"].values

    # Compute moral valence = sum(virtue - vice) across foundations
    required_cols = [f"{f}.virtue" for f in foundations] + [f"{f}.vice" for f in foundations]
    missing = [c for c in required_cols if c not in scored.columns]
    if missing:
        raise ValueError(
            f"{topic}: eMFD output missing columns {missing}. "
            f"Available: {list(scored.columns)}"
        )

    scored["moral_valence"] = sum(
        scored[f"{f}.virtue"] - scored[f"{f}.vice"] for f in foundations
    )

    # Normalise by moral density if available
    if "moral_nonmoral_ratio" in scored.columns:
        scored["moral_valence_norm"] = scored["moral_valence"] / (scored["moral_nonmoral_ratio"] + eps)
    else:
        scored["moral_valence_norm"] = scored["moral_valence"]

    # Summarise by gender
    summary = (
        scored.groupby("gender", dropna=False)
        .agg(
            n_sentences=("moral_valence", "size"),
            moral_valence_mean=("moral_valence", "mean"),
            moral_valence_sum=("moral_valence", "sum"),
            moral_valence_norm_mean=("moral_valence_norm", "mean"),
        )
    )
    print(summary)

    male_mean = float(summary.loc["M", "moral_valence_mean"]) if "M" in summary.index else 0.0
    female_mean = float(summary.loc["F", "moral_valence_mean"]) if "F" in summary.index else 0.0
    male_norm = float(summary.loc["M", "moral_valence_norm_mean"]) if "M" in summary.index else 0.0
    female_norm = float(summary.loc["F", "moral_valence_norm_mean"]) if "F" in summary.index else 0.0
    male_n = int(summary.loc["M", "n_sentences"]) if "M" in summary.index else 0
    female_n = int(summary.loc["F", "n_sentences"]) if "F" in summary.index else 0

    all_results.append({
        "topic": topic,
        "male_moral_valence_mean": male_mean,
        "female_moral_valence_mean": female_mean,
        "male_moral_valence_norm_mean": male_norm,
        "female_moral_valence_norm_mean": female_norm,
        "male_total": male_n,
        "female_total": female_n,
    })


Topic: 0_covid_virus_vaccine_pandemic


Processed: 0   0% |                      | Elapsed Time: 0:00:00 ETA:  --:--:--
Processed: 10   1% |                     | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 20   2% |                     | Elapsed Time: 0:00:00 ETA:   0:00:05
Processed: 39   4% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 50   6% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:05
Processed: 69   8% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 80  10% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 99  12% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 108  13% |❤❤                  | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 120  15% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 140  17% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 160  20% |❤❤❤❤                | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 170  21% |❤❤❤❤               

        n_sentences  moral_valence_mean  moral_valence_sum  \
gender                                                       
F               111            0.234320          26.009524   
M               676            0.293667         198.519048   

        moral_valence_norm_mean  
gender                           
F                      3.359862  
M                      4.385965  

Topic: 1_climate_emissions_carbon_energy


Processed: 0   0% |                      | Elapsed Time: 0:00:00 ETA:  --:--:--
Processed: 10   1% |                     | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 20   2% |                     | Elapsed Time: 0:00:00 ETA:   0:00:05
Processed: 30   3% |                     | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 39   5% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:05
Processed: 49   6% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 66   8% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 78  10% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 88  11% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 98  12% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 108  14% |❤❤                  | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 117  15% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 127  16% |❤❤❤                

        n_sentences  moral_valence_mean  moral_valence_sum  \
gender                                                       
F               118            0.349718          41.266667   
M               652            0.287372         187.366667   

        moral_valence_norm_mean  
gender                           
F                      5.714070  
M                      4.955735  

Topic: 2_scottish_scotland_snp_sturgeon


...

        n_sentences  moral_valence_mean  moral_valence_sum  \
gender                                                       
F                 6            0.000000           0.000000   
M                67            0.150178          10.061905   

        moral_valence_norm_mean  
gender                           
F                       0.00000  
M                       3.08936  

Topic: 8_nhs_health_doctors_patients


Processed: 0   0% |                      | Elapsed Time: 0:00:00 ETA:  --:--:--
Processed: 16   2% |                     | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 24   3% |                     | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 39   6% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 56   8% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 72  11% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 88  14% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 96  15% |❤❤❤                  | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 110  17% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 119  19% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 135  21% |❤❤❤❤                | Elapsed Time: 0:00:00 ETA:   0:00:02
Processed: 151  24% |❤❤❤❤                | Elapsed Time: 0:00:00 ETA:   0:00:02
Processed: 167  26% |❤❤❤❤❤              

        n_sentences  moral_valence_mean  moral_valence_sum  \
gender                                                       
F               121            0.420753          50.911111   
M               505            0.512422         258.773016   

        moral_valence_norm_mean  
gender                           
F                      6.383950  
M                      6.707616  

Topic: 9_eu_barnier_uk_deal


Processed: 0   0% |                      | Elapsed Time: 0:00:00 ETA:  --:--:--
Processed: 9   1% |                      | Elapsed Time: 0:00:00 ETA:   0:00:05
Processed: 17   2% |                     | Elapsed Time: 0:00:00 ETA:   0:00:06
Processed: 26   3% |                     | Elapsed Time: 0:00:00 ETA:   0:00:05
Processed: 43   6% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 60   8% |❤                    | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 76  11% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 85  12% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 94  14% |❤❤                   | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 102  15% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:04
Processed: 111  16% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 119  17% |❤❤❤                 | Elapsed Time: 0:00:00 ETA:   0:00:03
Processed: 128  19% |❤❤❤                

        n_sentences  moral_valence_mean  moral_valence_sum  \
gender                                                       
F               111            0.271772          30.166667   
M               558            0.306665         171.119048   

        moral_valence_norm_mean  
gender                           
F                      5.482032  
M                      6.690840  


In [9]:
results_df = pd.DataFrame(all_results)
print("\nAll topics — moral valence by gender:\n")
results_df


All topics — moral valence by gender:



,topic,male_moral_valence_mean,female_moral_valence_mean,male_moral_valence_norm_mean,female_moral_valence_norm_mean,male_total,female_total
0,0_covid_virus_vaccine_pandemic,0.293667,0.234320,4.385965,3.359862,676,111
1,1_climate_emissions_carbon_energy,0.287372,0.349718,4.955735,5.714070,652,118
2,2_scottish_scotland_snp_sturgeon,0.393759,0.364408,7.760480,7.185887,1281,194
3,3_biden_trump_sanders_democratic,0.222222,0.333333,3.415344,6.222222,42,9
4,4_schools_school_education_teachers,0.252956,0.254787,4.865949,5.587193,483,188
5,5_farage_ukip_nuttall_party,0.296059,0.282299,5.747906,5.573854,740,145
6,6_her_she_may_brexit,0.289284,0.288023,5.078376,5.467007,971,297
7,7_book_books_novel_my,0.150178,0.000000,3.089360,0.000000,67,6
8,8_nhs_health_doctors_patients,0.512422,0.420753,6.707616,6.383950,505,121
9,9_eu_barnier_uk_deal,0.306665,0.271772,6.690840,5.482032,558,111


In [10]:
results_path = out_dir / "morality_scores_by_topic.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved results to {results_path}")

Saved results to results_by_topic/morality_scores_by_topic.csv
